In [1]:
import re
import pandas as pd
import numpy as np
from alignment_interactive import process_dataframe, BatchReviewer, build_final_dataset

In [ ]:
def build_my_patterns_witness():
    name_chars = r"[^\s,]+"
    conj_akk   = r"(?:\s+(?:ù|and)\s+(?!KIŠIB)" + name_chars + r")?"
    gap_token  = r"(?:\s*<[^>]*>(?:\s+(?!IGI\b|DUMU\b|KIŠIB\b|KÙ\.BABBAR\b|witnessed\b|by\b)[^\s<]+)?)*"
    relation_akk = (
        r"(?:\s+(?:DUMU|a-ḫu-ú)\s+[^\s,;<]+(?:\s*<[^>]*>)*"
        r"(?:\s+(?!DUMU\b|IGI\b|a-ḫu-ú\b|a-ḫa-ma\b|a-na\b|i-na\b|§)[^\s,;]*-[^\s,;]+"
        r"(?:\s*<[^>]*>)*)*)*"
    )
    akk_name = r"[^\s]*[-\.][^\s]*[-\.][^\s]+|[^\s]*-[^\s]+(?:\s+(?!DUMU\b|IGI\b|a-ḫu-ú\b|§)[^\s]*-[^\s]+)*|[^\s]*\.[^\s]+"
    akk_igi_unit   = rf"IGI\s+(?!GÍR\b|a-we-li\b)(?:(?:{akk_name}){gap_token}(?:{relation_akk})|(?:DUMU|a-ḫu-ú)\s+{name_chars}{gap_token}{relation_akk}){conj_akk}\s*(?!\d)"
    akk_igi_multi  = rf"(?:{akk_igi_unit}\s*){{2,}}"
    akk_igi_single = rf"{akk_igi_unit}$"
    akk_igi        = rf"(?:{akk_igi_multi}|{akk_igi_single})"
    akk_kisib      = rf"(?:KIŠIB\s+{name_chars}{relation_akk}{conj_akk}\s*)+"
    akk_pattern    = rf"(?:{akk_kisib}|{akk_igi})"
    akk_re = re.compile(rf"({akk_pattern})", re.IGNORECASE | re.MULTILINE)

    occupations        = r"(?:packer|scribe|merchant|smith|weaver|potter|herald|priestess)"
    name_or_gap        = rf"(?:{name_chars}|{gap_token})"
    possessive_witness = r"(?:his|her)\s+(?:son|daughter|brother|sister|mother|wife|partner)"
    relation_eng = (
        r"(?:[,\s]+(?:"
        r"(?:his|her)\s+(?:brother|sister|son|daughter|mother|father|wife|partner)"
        r"|the\s+[^\s,;.]+(?:\s+of\s+" + name_or_gap + r")?"
        r"|(?:younger|elder|older)\s+(?:son|daughter)"
        r"|[^\s,;]+'s\s+(?:younger|elder|older)?\s*(?:son|daughter|" + occupations + r")"
        r"|(?:s\.|son|\(grand\)son|grandson|brother|sister|mother|father|daughter|wife|" + occupations + r")\s+of\s+" + gap_token + name_chars +
        r"))*"
    )
    conj_eng = (
        r"(?:"
        r"(?:,\s*(?!(?:and|ù|by)\s|<[^>]+>)" + name_chars + r")*"
        r"(?:,?\s*(?:and|ù)\s+(?!witnessed\s+by|by\s)" + name_chars + r")?"
        r")"
    )
    possessive_prefix = r"(?:(?:his|her)\s+(?:son|daughter|brother|sister|mother|wife)\s+)?"
    eng_unit = (
        rf"witnessed\s+by\s+"
        rf"(?:{possessive_witness}|{possessive_prefix}{name_chars})"
        rf"{gap_token}{relation_eng}{conj_eng}"
    )
    trailing_collective = r"(?:,\s*the\s+(?:sons|daughters|brothers|sisters)\s+of\s+" + name_or_gap + r")?"
    eng_pattern = (
        rf"{eng_unit}"
        rf"(?:[\s,;]+(?:and\s+)?(?:witnessed\s+)?by\s+"
        rf"(?:{possessive_witness}|{possessive_prefix}(?:(?![,;]){name_chars})?)"
        rf"{gap_token}{relation_eng}{conj_eng}{gap_token})*"
        rf"{trailing_collective}{gap_token}[,;]?"
    )
    eng_full = rf'(^\s*|\n\s*|[.,;:"\u201C\u201D]\s*|<[^>]*>\s*)(?=witnessed\s+by)({eng_pattern})'
    eng_re = re.compile(eng_full, re.IGNORECASE | re.MULTILINE)

    return akk_re, eng_re


def build_my_patterns_sealed_by():
    name_chars = r"[^\s,]+"
    
    # AKK
    conj_akk     = r"(?:\s+(?:ù|and)\s+(?!KIŠIB)" + name_chars + r")?"
    relation_akk = r"(?:\s+(?:DUMU|a-ḫu-ú)\s+" + name_chars + r")*"
    akk_pattern  = rf"(?:KIŠIB\s+{name_chars}{relation_akk}{conj_akk}\s*)+"
    # -ENGLISH
    relation_eng = r"(?:[,\s]+(?:(?:his|her)\s+(?:brother|sister)|the\s+scribe|(?:s\.|son|brother|sister|daughter|wife|\(grand\)son|grandson)\s+of\s+" + name_chars + r"))?"    
    conj_eng     = r"(?:\s+(?:and|ù)\s+(?!sealed\s+by)" + name_chars + r")?"
    trailing_collective = r"(?:,\s*the\s+(?:sons|daughters|brothers|sisters)\s+of\s+" + name_chars + r")?"

    possessive_prefix = r"(?:(?:his|her)\s+(?:son|daughter|brother|sister)\s+)?"
    eng_unit = rf"sealed\s+by\s+{possessive_prefix}{name_chars}{relation_eng}{conj_eng}"
    eng_pattern = rf"{eng_unit}(?:[\s,;]+(?:and\s+)?(?:sealed\s+)?by\s+{possessive_prefix}(?:(?![,;]){name_chars})?{relation_eng}{conj_eng})*{trailing_collective}[,;]?"

    eng_full = rf"(^\s*|[,;:]\s*)({eng_pattern})"

    akk_re = re.compile(rf"({akk_pattern})", re.IGNORECASE | re.MULTILINE)

    eng_re = re.compile(eng_full, re.IGNORECASE | re.MULTILINE)

    return akk_re, eng_re

def build_my_patterns_seal_of():
    name_chars   = r"[^\s]+"
    conj_akk     = r"(?:\s+(?:ù|and)\s+(?!KIŠIB)" + name_chars + r")?"
    relation_akk = r"(?:\s+(?:DUMU|a-ḫu-ú)\s+" + name_chars + r")?"
    akk_pattern  = rf"(?:KIŠIB\s+{name_chars}{relation_akk}{conj_akk}\s*)+"

    relation_eng = r"(?:[,\s]+(?:s\.|son|brother|sister|daughter|wife)\s+of\s+" + name_chars + r")?"
    conj_eng     = r"(?:\s+(?:and|ù)\s+(?!seal\s+of)" + name_chars + r")?"
    eng_unit     = rf"seal\s+of\s+{name_chars}{relation_eng}{conj_eng}"
    eng_pattern  = rf"{eng_unit}(?:[\s,;]+(?:and\s+)?seal\s+of(?:\s+(?![,;]){name_chars})?{relation_eng}{conj_eng})*[,;]?"
    eng_full     = rf"(^|[.,;:]\s*)({eng_pattern})"

    akk_re = re.compile(rf"({akk_pattern})", re.IGNORECASE | re.MULTILINE)
    eng_re = re.compile(eng_full,            re.IGNORECASE | re.MULTILINE)
    return akk_re, eng_re

def build_my_patterns_by_hand():
    match_all_akk = re.compile(r'(.+)', re.DOTALL)
    match_all_eng = re.compile(r'()(.+)', re.DOTALL)  # group 1 = empty prefix, group 2 = everything

    return match_all_akk, match_all_eng


def build_my_patterns_kanesh_colony():
    """
    Matches the juridical oath formula:

    EN: "The Kanesh colony gave us ... Assur"
    AK: a-na a-wa-tim a-ni-a-tim kà-ru-um kà-ni-iš i-dí-ni-a-tí-ma
        ṭup-pá-am ší-bu-tí-ni IGI GÍR ša a-šùr ni-dí-in

    Five fixed anchors; everything between them floats so both attested
    word-orders for ší-bu-tí-ni (before or after IGI GÍR) are covered.
    """

    gap_token = r"(?:\s*<[^>]*>(?:\s+[^\s<]+)?)*"

    # AKK
    ana_awatim    = r"a-na\s+a-wa-tim\s+a-ni-[ai]-tim"   # "for these matters"
    karum_kanis   = r"kà?-ru-um\s+kà?-ni-iš"             # "the Kanesh kārum"
    iddiniatima   = r"i-dí-ni-a-tí-ma"                   # "gave us"
    igi_gir_assur = r"IGI\s+GÍR\s+ša\s+a-šù?r"          # "before the sword of Aššur"
    nidin         = r"ni-dí-in"                           # "we gave"

    before_igi = r"(?:\s+(?!IGI\b)\S+)*"
    after_igi  = r"(?:\s+(?!ni-dí-in\b)\S+)*"

    akk_pattern = (
        rf"{ana_awatim}\s+"
        rf"{karum_kanis}\s+"
        rf"{iddiniatima}"
        rf"{before_igi}\s+"
        rf"{igi_gir_assur}"
        rf"{after_igi}\s+"
        rf"{nidin}"
    )

    # ENGLISH
    eng_pattern = (
        r"[Tt]he\s+Kanesh\s+colony\s+gave\s+us"
        r"(?:\s+[^.;\n]*?)?"                        # optional intervening text
        r"\s+(?:[Aa]šš?ur|[Aa]ssur)"               # ends at Assur / Aššur
    )
    eng_full = rf'(^\s*|\n\s*|[.,;:"\u201C\u201D]\s*)({eng_pattern})'

    akk_re = re.compile(rf"({akk_pattern})", re.IGNORECASE | re.MULTILINE)
    eng_re = re.compile(eng_full,             re.IGNORECASE | re.MULTILINE)

    return akk_re, eng_re

In [3]:
df = pd.read_csv("../../dataset/maas_aligned.csv")

In [ ]:

akk_re, eng_re = build_my_patterns_kanesh_colony()

ratio = 1 # Ratio between words in akkadian and english
pos_threshold = 0.15 # Irrelevant for simple patterns

clean, flagged = process_dataframe(
    df,
    translit_col="transliteration",
    transl_col="translation",
    substring='The Kanesh colony gave us', # match es if found in Akkadian OR English
    akk_pattern=akk_re, akk_group=1,
    eng_pattern=eng_re, eng_group=2,
    ratio_min=1/ratio,   # override here
    ratio_max=ratio,   # override here
    pos_threshold = 0.15,
    must_match_akkadian = False,
    must_match_english = False,
)

Rows matching 'The Kanesh colony gave us': 47
Processed 47 rows — 11 clean  |  36 flagged


In [5]:
# Step 2 — review flagged items interactively
reviewer = BatchReviewer(flagged)
reviewer.show()

In [6]:

all_results = clean + flagged
df = pd.DataFrame(build_final_dataset(all_results))  # list of {'row_index', 'segment_idx', 'akk', 'eng'}
df["transliteration"] = df["akk"]
df["translation"] = df["eng"]
df.reset_index(inplace=True)
df.rename(columns={"index": "id"}, inplace=True)

In [7]:
df.to_csv("maas_aligned-v2.csv")

In [8]:
def drop_empty_rows(df, translit_col="transliteration", transl_col="translation"):
    mask = (
        df[translit_col].isna() | (df[translit_col] == "") |
        df[transl_col].isna()   | (df[transl_col] == "")
    )
    print(len(df))
    return df[~mask].reset_index(drop=True)

In [9]:
df = pd.read_csv("maas_aligned-v2.csv")
df = drop_empty_rows(df)
df.to_csv("maas_aligned-v3.csv")


69


In [10]:
df1 = pd.read_csv("../../dataset/maas_aligned.csv")
df2 = pd.read_csv("maas_aligned-v2.csv")
df = pd.concat([df1, df2], ignore_index=True)
df.to_csv("maas_aligned_v2.csv")